# Setup

In [45]:
# Optional autoreload while developing locally

%load_ext autoreload
%autoreload 2

In [46]:

# 0) Local-only setup and base diagnostics
import os
import sys
import platform
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())

PROJECT_DIR = Path.cwd()



In [47]:

# 0.3) Shared device diagnostics
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))
    DEVICE = "cuda:0"
else:
    DEVICE = "cpu"
print("Using DEVICE =", DEVICE)


# Imports

In [48]:
import pickle
import re

import pandas as pd
from spacy.tokens import Doc


from tokenizer import create_tokenizer, ensure_booknlp_extensions, tokens_to_dicts
from doc_chunker import create_chunker
from runtime_config import (
    RUNTIME_PROFILE,
    DEVICE,
    CHUNK_SIZE,
    OVERLAP_SENTENCES,
    MAX_EXPANDED_CHUNK_TOKENS,
)
from coreference.coreference_sub_orchestrator import (
    run_coreference_resolution,
    print_non_singleton_coref_clusters,
)

from coreference.coref_schema import require_coref_layer


# Functions

In [49]:
# text loading + preprocessing
def load_txt(path):
    path = Path(path)

    try:
        return path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return path.read_text(encoding="latin-1")

In [50]:
def preprocess_for_NER(text):
    # Normalize Windows/Mac newlines
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove possible invisible BOM
    text = text.replace("\ufeff", "")

    # Normalize curly apostrophes/quotes
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')

    # Replace line breaks inside paragraphs with spaces,
    # but preserve paragraph boundaries
    paragraphs = text.split("\n\n")
    paragraphs = [
        re.sub(r"\s*\n\s*", " ", paragraph).strip()
        for paragraph in paragraphs
        if paragraph.strip()
    ]

    text = "\n\n".join(paragraphs)

    chapter_re = re.compile(
        r"(?im)^\s*chapter\s+(?:[ivxlcdm]+|\d+)\b[^\n]*\n*"
    )

    text = chapter_re.sub("", text)

    # Normalize multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [51]:

def resolve_text_path(
    *,
    project_dir: Path,
    filename: str = "oz_full.txt",
    local_path: str | Path = "../../datasets/libri/oz_full.txt",
) -> Path:
    """Resolve the local input text path."""
    text_path = Path(local_path)
    if text_path.exists():
        return text_path

    fallback_path = project_dir / filename
    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(
        "Could not resolve a local text file. "
        f"Tried LOCAL_TEXT_PATH={text_path} and PROJECT_DIR / {filename!r} = {fallback_path}."
    )


In [52]:
def save_doc(doc, path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("wb") as f:
        pickle.dump(doc, f)
    return path


def load_doc(path: str | Path):
    # BookNLP/tokenizer extensions must be registered before unpickling the Doc.
    # The global coreference extension is registered inside the coreference
    # sub-orchestrator immediately before final annotation.
    ensure_booknlp_extensions()
    path = Path(path)
    with path.open("rb") as f:
        return pickle.load(f)


# Config

## I/O config

In [53]:
# Requested local file
TEXT_FILENAME = "oz_full.txt"
LOCAL_TEXT_PATH = Path(f"../../datasets/libri/{TEXT_FILENAME}")

TEXT_PATH = resolve_text_path(
    project_dir=PROJECT_DIR,
    filename=TEXT_FILENAME,
    local_path=LOCAL_TEXT_PATH,
)

OUTPUT_ROOT = PROJECT_DIR / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

TOKENIZED_DOC_PATH = OUTPUT_ROOT / "tokenized_doc.pkl"
ANNOTATED_DOC_PATH = OUTPUT_ROOT / "annotated_doc.pkl"

print("TEXT_PATH =", TEXT_PATH)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("TOKENIZED_DOC_PATH =", TOKENIZED_DOC_PATH)
print("ANNOTATED_DOC_PATH =", ANNOTATED_DOC_PATH)


In [54]:
GLOBAL_COREF_OUTPUT_DIR = OUTPUT_ROOT / "global_coref"
GLOBAL_COREF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)

## Runtime and pipeline config

In [55]:
# Pipeline configuration
SPACY_MODEL = "en_core_web_sm"
TOKENIZER_DISABLE = ("ner",)

print("RUNTIME_PROFILE =", RUNTIME_PROFILE)
print("DEVICE =", DEVICE)
print("CHUNK_SIZE(expanded) =", CHUNK_SIZE)
print("OVERLAP_SENTENCES =", OVERLAP_SENTENCES)
print("MAX_EXPANDED_CHUNK_TOKENS =", MAX_EXPANDED_CHUNK_TOKENS)
print("GLOBAL_COREF_OUTPUT_DIR =", GLOBAL_COREF_OUTPUT_DIR)


# Main

### Tokenization

In [56]:
# Load or create the stable tokenized Doc artifact.
if TOKENIZED_DOC_PATH.exists():
    print(f"[doc] Loading tokenized Doc from {TOKENIZED_DOC_PATH}")
    doc = load_doc(TOKENIZED_DOC_PATH)
else:
    print("[doc] Tokenized Doc not found; creating it from raw text.")
    raw_text = load_txt(TEXT_PATH)
    text = preprocess_for_NER(raw_text)
    tokenizer = create_tokenizer(
        SPACY_MODEL,
        disable=TOKENIZER_DISABLE,
        verbose=True,
    )
    doc = tokenizer.tokenize(text)
    save_doc(doc, TOKENIZED_DOC_PATH)
    print(f"[doc] Saved tokenized Doc to {TOKENIZED_DOC_PATH}")

print("Doc tokens:", len(doc))
print("Doc sentences:", sum(1 for _ in doc.sents))

In [57]:
# df_tokens = pd.DataFrame(tokens_to_dicts(doc))
# df_tokens.head()

### Chunking

In [58]:
chunker = create_chunker(
    chunk_size=CHUNK_SIZE,
    overlap_sentences=OVERLAP_SENTENCES,
    max_expanded_chunk_tokens=MAX_EXPANDED_CHUNK_TOKENS,
)
chunk_plan = chunker.plan(doc)

In [59]:
# print(f"Planned {len(chunk_plan)} chunks without materializing chunk docs.")
# for spec in chunk_plan.specs[:5]:
#     print(
#         spec.chunk_id,
#         "expanded=", (spec.global_start, spec.global_end),
#         "expanded_tokens=", spec.n_tokens,
#         "core=", (spec.core_start, spec.core_end),
#         "left_overlap=", (spec.left_overlap_start, spec.left_overlap_end),
#         "right_overlap=", (spec.right_overlap_start, spec.right_overlap_end),
#     )


### Coreference clusters extraction

In [60]:
if ANNOTATED_DOC_PATH.exists():
    print(f"[coref-annotation] Loading coref-annotated Doc from {ANNOTATED_DOC_PATH}")
    doc = load_doc(ANNOTATED_DOC_PATH)
else:
    print("[coref-annotation] Annotated Doc not found; calling annotator sub-orchestrator.")
    doc = run_coreference_resolution(
        doc=doc,
        chunker=chunker,
        chunk_plan=chunk_plan,
        document_id=TEXT_PATH.stem,
        output_dir=GLOBAL_COREF_OUTPUT_DIR,
    )
    save_doc(doc, ANNOTATED_DOC_PATH)
    print("[coref-annotation] Saved annotated Doc to:", ANNOTATED_DOC_PATH)
    
    
if not Doc.has_extension("coref_layer"):
    Doc.set_extension("coref_layer", default=None)

In [61]:
print("[coref-annotation] Summary:")
print(doc._.coref_layer.summary())
print_non_singleton_coref_clusters(doc)

### Coreference cluster exploration

###### from the package

In [ ]:
cluster_ids = [8, 97, 153]
n_mentions = None  # None = score ALL mentions for each requested cluster

random_seed = 67  # for reproducing random behaviour

from character_ocean_scoring import (
    ContextConfig,
    MentionRenderingConfig,
    OceanScoringConfig,
    OceanWeightConfig,
    DirectNLIConfig,
    benchmark_ocean_clusters_probability_csvs,
)

dfs_by_cluster_id = benchmark_ocean_clusters_probability_csvs(
    doc,
    cluster_ids=cluster_ids,
    n_mentions=n_mentions,

    # Marathon output folder
    output_dir="./outputs/OCEAN_profiles",

    context_config=ContextConfig(
        n_sentences_before=0,
        n_sentences_after=0,
        mark_mention=True,
        deduplicate=False,
    ),
    rendering_config=MentionRenderingConfig(
        canonicalize_simple_mentions=True,
        keep_first_second_person=True,
    ),
    scoring_config=OceanScoringConfig(
        subject_aware=True,
    ),
    weight_config=OceanWeightConfig(
        enabled=True,
    ),
    nli_config=DirectNLIConfig(
        pair_batch_size=16,
    ),

    # Conservative for old PCs.
    chunk_size=8,

    # Marathon/resume behavior.
    overwrite_csv=False,
    resume_from_csv=True,
    use_sqlite_cache=True,

    random_seed=random_seed,
    sort_sample_by_cluster_order=True,
    print_progress=True,

    # Saves RAM: results are written to CSV; do not keep all DataFrames in memory.
    return_dataframes=False,
)

# Reload only the cluster you want to inspect.
import pandas as pd
from pathlib import Path

output_dir = Path("./outputs/OCEAN_profiles")

csv_files = sorted(output_dir.glob("OCEAN_scores_cluster_8_*_all.csv"))
if not csv_files:
    raise FileNotFoundError("No CSV found for cluster_id=8 in ./outputs/OCEAN_profiles")

df = pd.read_csv(csv_files[-1])
df.head()

CSV-first sampled multi-cluster OCEAN run
cluster_ids: [8, 97, 153]
requested mentions per cluster: None
output_dir: outputs\OCEAN_profiles
overwrite_csv: False
resume_from_csv: True
use_sqlite_cache: True
return_dataframes: False
random_seed: 67
sort_sample_by_cluster_order: True
device: cuda: NVIDIA GeForce GTX 1050 with Max-Q Design

----------------------------------------------------------------------------------------------------
cluster 1/3
cluster_id: 8
subject from doc._.coref_layer: 'Dorothy'
csv_path: outputs\OCEAN_profiles\OCEAN_scores_cluster_8_Dorothy_all.csv
per_cluster_seed: 8621439430968330030
----------------------------------------------------------------------------------------------------
CSV-first OCEAN probability benchmark
cluster_id: 8
subject: 'Dorothy'
requested mentions: None
sample_mentions: True
random_seed: 8621439430968330030
start_index: 0
total mentions in cluster: 1900
device: cuda: NVIDIA GeForce GTX 1050 with Max-Q Design
chunk_size: 8
pair_batch_si

In [ ]:
OCEAN_TRAITS = (
    "openness",
    "conscientiousness",
    "extraversion",
    "agreeableness",
    "neuroticism",
)


def weighted_ocean_profile_from_csv(
    csv_path: str,
    *,
    trait_columns: tuple[str, ...] = OCEAN_TRAITS,
    weight_column: str = "OCEAN_weight",
) -> dict[str, float]:
    """
    Read an OCEAN mention-level CSV and compute the weighted average
    of each OCEAN trait using OCEAN_weight as the weight.

    INPUT:
        csv_path:
            Path to a CSV with columns:
                openness
                conscientiousness
                extraversion
                agreeableness
                neuroticism
                OCEAN_weight

    OUTPUT:
        {
            "openness": float,
            "conscientiousness": float,
            "extraversion": float,
            "agreeableness": float,
            "neuroticism": float,
        }
    """

    df = pd.read_csv(csv_path)

    required_columns = set(trait_columns) | {weight_column}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"CSV is missing required columns: {sorted(missing_columns)}"
        )

    working_df = df[list(trait_columns) + [weight_column]].copy()

    for column in trait_columns + (weight_column,):
        working_df[column] = pd.to_numeric(working_df[column], errors="coerce")

    working_df = working_df.dropna(subset=list(trait_columns) + [weight_column])

    if working_df.empty:
        raise ValueError("No valid numeric rows found after cleaning the CSV.")

    weights = working_df[weight_column].clip(lower=0.0)
    total_weight = weights.sum()

    if total_weight <= 1e-9:
        raise ValueError(
            f"Total {weight_column} is zero; weighted average cannot be computed."
        )

    avg_traits = {
        trait: round(
            float((working_df[trait] * weights).sum() / total_weight),
            2,
        )
        for trait in trait_columns
    }

    return avg_traits

In [ ]:
print("Dorothy")
avg_traits = weighted_ocean_profile_from_csv("./outputs/OCEAN_scores_cluster_8_Dorothy_100.csv",)
print(avg_traits)

print("\nthe Lion")
avg_traits = weighted_ocean_profile_from_csv("./outputs/OCEAN_scores_cluster_153_the_Lion_100.csv",)
print(avg_traits)

print("\nthe Scarecrow")
avg_traits = weighted_ocean_profile_from_csv("./outputs/OCEAN_scores_cluster_97_the_Scarecrow_100.csv")
print(avg_traits)